# Assignment 4 — Exercise 1

In [34]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import least_squares
from scipy.stats import norm


SEED = 42
plt.style.use("seaborn-v0_8")

calibrated_params = pd.read_json("SABR_calibrated_params.json")
calibrated_params

,calibrated_value
sigma0,0.294404
beta,1.000000
rho0,-1.000000
alpha0,0.405636
a,0.000001
b,0.091122


In [35]:
S0 = 100
K = 98
r = 0
q = 0.02
Ts = np.array([0.0, 0.25, 0.5, 0.75, 1])
deltat = 0.25

### Constant volatility

In the following part, we apply a constant volatility equal to 23%. We build the paths by simulating the stock price paths only at the 4 exercise dates.

In [36]:
# Constant volatility
sigma = 0.23
N = 1_000_000

exp = lambda z: np.exp((r-q-sigma**2 / 2) * deltat + sigma * np.sqrt(deltat) * z)

# Simulating a N * 4 matrix for S (N simulated paths, GBM)
S = np.full((N, len(Ts)), np.nan)
S[:, 0] = S0
for i in range(1, len(Ts)):
    z = norm.rvs(loc=0, scale=1, size=N, random_state=SEED)
    S[:, i] = S[:, i-1] * exp(z)
S = np.delete(S, [0], axis=1)

# Deriving the A matrix for each path
A = np.cumsum(S, axis=1) / np.arange(1, S.shape[1] + 1)

A.shape, S.shape

((1000000, 4), (1000000, 4))

In [37]:
# Immediate payoff matrix
P = np.maximum(A - K, 0.0)

# Initialize the cashflow
cashflow = P[:, -1].copy()
t_idx = np.full(len(S), S.shape[1] - 1, dtype=int)
times = Ts[1:]

for k in np.arange(0, S.shape[1]-1, 1)[::-1]:
    itm = P[:, k] > 0

    s = S[itm, k]
    a = A[itm, k]

    X = np.column_stack([np.ones_like(s), s, s**2, s**3, a, a**2, a**3])
    Y = cashflow[itm] * np.exp(-r * (times[t_idx[itm]] - times[k]))

    beta, _, _, _ = np.linalg.lstsq(X, Y)
    C_hat = X @ beta

    exercise = P[itm, k] > C_hat
    idx = np.where(itm)[0][exercise]
    cashflow[idx] = P[idx, k]
    t_idx[idx] = k


pv = cashflow * np.exp(-r * times[t_idx])
price = pv.mean()
se = cashflow.std(ddof=1) / np.sqrt(N)

print(f"Price of the option is {price.round(3)} and its standard error is {se.round(3)}\nover {N} simulated paths")

Price of the option is 13.654 and its standard error is 0.023
over 1000000 simulated paths


### SABR stochastic volatility model

We now use the calibrated SABR model from assignment 3. The parameters were imported in a JSON file and displayed in the first cell. We first build paths by simulating the stock price at shorter intervals (weekly) and then use an Euler scheme for the simulations.

In [38]:
# Load SABR parameters and define the time grid
params = calibrated_params["calibrated_value"].to_dict()
rng = np.random.default_rng(SEED)
dt = 1 / 52
times = Ts[1:]
steps = np.rint(times / dt).astype(int)

# Simulate SABR paths
F = np.full(N, float(S0))
sigma = np.full(N, params["sigma0"])
spots = np.zeros((N, len(times)))
previous_step = 0
for k, step_end in enumerate(steps):
    for j in range(previous_step, step_end):
        t = j * dt
        rho = params["rho0"] * np.exp(-params["a"] * t)
        alpha = params["alpha0"] * np.exp(-params["b"] * t)
        z1 = rng.normal(size=N)
        z2 = rho * z1 + np.sqrt(1 - rho**2) * rng.normal(size=N)
        F = np.maximum(F + sigma * F**params["beta"] * np.sqrt(dt) * z1, 0)
        sigma = sigma * (1 + alpha * np.sqrt(dt) * z2)
    spots[:, k] = F * np.exp((r - q) * times[k])
    previous_step = step_end

In [39]:
# Asian call payoffs
averages = np.cumsum(spots, axis=1) / np.arange(1, len(times) + 1)
payoffs = np.maximum(averages - K, 0)

# Longstaff-Schwartz
cashflows = payoffs[:, -1].copy()
exercise = np.full(N, len(times) - 1)
for k in range(len(times) - 2, -1, -1):
    itm = payoffs[:, k] > 0
    if not np.any(itm):
        continue

    S = spots[itm, k] / S0
    A = averages[itm, k] / S0
    X = np.column_stack([S**0, S, S**2, S**3, A, A**2, A**3])
    Y = cashflows[itm] * np.exp(-r * (times[exercise[itm]] - times[k]))

    coeff = np.linalg.lstsq(X, Y, rcond=None)[0]
    continuation = X @ coeff
    exercise_now = np.where(itm)[0][payoffs[itm, k] > continuation]
    cashflows[exercise_now] = payoffs[exercise_now, k]
    exercise[exercise_now] = k

In [40]:
# Price and standard error
pv = cashflows * np.exp(-r * times[exercise])
price = np.mean(pv)
se = np.std(pv, ddof=1) / np.sqrt(N)
print(f"SABR option price: {price:.4f}")
print(f"Monte Carlo standard error: {se:.4f}")

SABR option price: 8.5568
Monte Carlo standard error: 0.0109
